In [17]:
import yfinance as yf
import pandas as pd

df = yf.download(
    tickers="GC=F",
    interval="1d",
    start="2025-12-01",
    end="2026-01-01",
    progress=False
)


df.reset_index(inplace=True)

# flatten multi-index columns
df.columns = df.columns.get_level_values(0)

# ensure datetime
df["Date"] = pd.to_datetime(df["Date"])

# add day name
df["day_name"] = df["Date"].dt.day_name()

# add candle direction
df["candle_body"] = (df["Close"] - df["Open"]) / 0.01

# add price movement
df["price_movement_pips"] = (df["High"] - df["Low"]) / 0.01

# group by day_name and sum absolute movement
daily_total = df.groupby("day_name")["price_movement_pips"].apply(
    lambda x: x.abs().sum())
# sort descending to get most active weekdays first
daily_total = daily_total.sort_values(ascending=False)


report = daily_total.reset_index()
report.columns = ["Weekday", "Total_Movement_Pips"]
report["Rank"] = report["Total_Movement_Pips"].rank(
    ascending=False, method="dense").astype(int)
report = report.sort_values("Rank")

df["month"] = df["Date"].dt.month_name()
report["Month"] = df["month"].iloc[0]
report = report[["Month", "Weekday", "Total_Movement_Pips", "Rank"]]


print(report)



month_name = report["Month"].iloc[0]
year = df["Date"].dt.year.iloc[0]

filename = f"gold_{month_name.lower()}_{year}_weekday_movement.csv"


report.to_csv(filename, index=False)

      Month    Weekday  Total_Movement_Pips  Rank
0  December    Tuesday         27539.990234     1
1  December     Monday         27420.019531     2
2  December  Wednesday         24599.951172     3
3  December     Friday         22179.980469     4
4  December   Thursday         12929.980469     5


In [20]:
import pandas as pd
from pathlib import Path

# ==============================
# CONFIG
# ==============================
DATA_DIR = Path(".")   # <-- your folder
OUTPUT_FILE = "gold_2025_weekday_movement_all_months.csv"

MONTH_ORDER = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

# ==============================
# LOAD CSV FILES
# ==============================
csv_files = sorted(DATA_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("❌ No CSV files found in gold_data folder!")

# ==============================
# MERGE ALL MONTHS
# ==============================
df_all = pd.concat(
    (pd.read_csv(file) for file in csv_files),
    ignore_index=True
)

# ==============================
# DATA VALIDATION
# ==============================
required_cols = {"Weekday", "Total_Movement_Pips", "Rank", "Month"}
missing = required_cols - set(df_all.columns)

if missing:
    raise ValueError(f"❌ Missing columns: {missing}")

# ==============================
# SORT LOGIC
# ==============================
df_all["Month"] = pd.Categorical(
    df_all["Month"],
    categories=MONTH_ORDER,
    ordered=True
)

df_all = df_all.sort_values(
    by=["Month", "Rank"],
    ascending=[True, True]
).reset_index(drop=True)

# ==============================
# EXPORT FINAL FILE
# ==============================
df_all.to_csv(OUTPUT_FILE, index=False)

print("✅ Combined yearly gold report created")
print(f"📁 Saved as: {OUTPUT_FILE}")
print(f"📊 Total rows: {len(df_all)}")

✅ Combined yearly gold report created
📁 Saved as: gold_2025_weekday_movement_all_months.csv
📊 Total rows: 60
